# SynthDoG `SynthDoG_en_pdfs_v1000` — dataset shape analysis

Analyzes the 1,000-sample dataset generated from `config_en-pdfs.yaml` against the 2M-doc
FinePDFs corpus (804 train / 89 validation / 107 test, content-hash split).

Covers image geometry, layout/zones, fonts, paper color, quality metrics, and the per-sample
effect provenance — both as a sanity check on the generator and as a reference for
interpreting downstream OCR model errors.

Run `uv sync --group analysis` before executing this notebook.


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

REPO_ROOT = Path("/Users/hunterheidenreich/git/GutenOCR/data/synthdog_grounding")
sys.path.insert(0, str(REPO_ROOT))

import analyze
from serialization import SPLITS, decode_metadata

DATA_ROOT = REPO_ROOT / "outputs" / "SynthDoG_en_pdfs_v1000"

plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.facecolor"] = "white"
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

## 1. Loader

One pass per split over `metadata.jsonl`, decoded via `serialization.decode_metadata`.
No image files are opened here — `generation_params.canvas_size` gives width/height for
free, avoiding 1,000 unnecessary PIL opens (which is what `data_readers.iter_samples`
would otherwise do). This also captures `generation_params` in full, which `iter_samples`
drops entirely — needed for the font/color/effects provenance analysis below.

Produces three DataFrames at three different grains:
- `samples_df` — one row per sample (quality metrics + generation params + flattened effects)
- `lines_df` — one row per text line (font, size, color, geometry)
- `blocks_df` — one row per text block (region type, geometry)


In [ ]:
def load_split(split_dir: Path, split_name: str):
    meta_path = split_dir / "metadata.jsonl"
    sample_rows, line_rows, block_rows, effects_rows = [], [], [], []

    with meta_path.open("r", encoding="utf-8") as f:
        for raw in f:
            raw = raw.strip()
            if not raw:
                continue
            rec = json.loads(raw)
            gt = decode_metadata(rec)

            qm = gt.get("quality_metrics", {})
            gp = gt.get("generation_params", {})
            lines = gt.get("text_lines", [])
            blocks = gt.get("text_blocks", [])

            sample_id = Path(rec["file_name"]).stem
            canvas_w, canvas_h = gp.get("canvas_size", [None, None])
            paper_s = gp.get("paper_rgb_sampled") or [None, None, None]
            paper_r = gp.get("paper_rgb_rendered") or [None, None, None]
            med_rgb = gp.get("text_color_median_rgb") or [None, None, None]
            body_col_counts = gp.get("body_col_counts", []) or []

            row = {
                "sample_id": sample_id,
                "split": split_name,
                "file_name": rec["file_name"],
                "n_lines": len(lines),
                "n_blocks": len(blocks),
                **qm,
                "landscape": gp.get("landscape"),
                "canvas_w": canvas_w,
                "canvas_h": canvas_h,
                "jpeg_quality": gp.get("jpeg_quality"),
                "gp_skew_angle": gp.get("skew_angle"),
                "paper_r_sampled": paper_s[0],
                "paper_g_sampled": paper_s[1],
                "paper_b_sampled": paper_s[2],
                "paper_r_rendered": paper_r[0],
                "paper_g_rendered": paper_r[1],
                "paper_b_rendered": paper_r[2],
                "stain_applied": gp.get("stain_applied"),
                "paper_luminance": gp.get("paper_luminance"),
                "text_color_mode": gp.get("text_color_mode"),
                "med_color_r": med_rgb[0],
                "med_color_g": med_rgb[1],
                "med_color_b": med_rgb[2],
                "zones_rendered": tuple(sorted(gp.get("zones_rendered", []) or [])),
                "body_grid_count": gp.get("body_grid_count"),
                "body_col_counts": tuple(body_col_counts),
            }
            sample_rows.append(row)

            effects = gp.get("effects", {}) or {}
            eff_row = {"sample_id": sample_id, "split": split_name}
            for eff_name, eff_val in effects.items():
                for k, v in (eff_val or {}).items():
                    eff_row[f"{eff_name}.{k}"] = v
            effects_rows.append(eff_row)

            for ln in lines:
                bbox = ln.get("bbox") or [None, None, None, None]
                tc = ln.get("text_color_rgb") or [None, None, None]
                line_rows.append(
                    {
                        "sample_id": sample_id,
                        "split": split_name,
                        "line_id": ln.get("line_id"),
                        "block_id": ln.get("block_id"),
                        "text_len": len(ln.get("text", "")),
                        "x1": bbox[0],
                        "y1": bbox[1],
                        "x2": bbox[2],
                        "y2": bbox[3],
                        "has_quad": ln.get("quad") is not None,
                        "font_family": ln.get("font_family"),
                        "font_size_px": ln.get("font_size_px"),
                        "line_color_r": tc[0],
                        "line_color_g": tc[1],
                        "line_color_b": tc[2],
                    }
                )

            for blk in blocks:
                bbox = blk.get("bbox") or [None, None, None, None]
                block_rows.append(
                    {
                        "sample_id": sample_id,
                        "split": split_name,
                        "block_id": blk.get("block_id"),
                        "region_type": blk.get("region_type", "body"),
                        "n_lines": len(blk.get("line_ids", []) or []),
                        "x1": bbox[0],
                        "y1": bbox[1],
                        "x2": bbox[2],
                        "y2": bbox[3],
                    }
                )

    return (
        pd.DataFrame(sample_rows),
        pd.DataFrame(line_rows),
        pd.DataFrame(block_rows),
        pd.DataFrame(effects_rows),
    )


samples_parts, lines_parts, blocks_parts, effects_parts = [], [], [], []
for split in SPLITS:
    split_dir = DATA_ROOT / split
    if not split_dir.exists():
        continue
    s, ln, b, e = load_split(split_dir, split)
    samples_parts.append(s)
    lines_parts.append(ln)
    blocks_parts.append(b)
    effects_parts.append(e)

samples_df = pd.concat(samples_parts, ignore_index=True)
lines_df = pd.concat(lines_parts, ignore_index=True)
blocks_df = pd.concat(blocks_parts, ignore_index=True)
effects_df = pd.concat(effects_parts, ignore_index=True)

# Merge flattened effect columns onto samples_df (one row per sample already).
samples_df = samples_df.merge(effects_df, on=["sample_id", "split"], how="left")
samples_df["aspect_ratio"] = samples_df["canvas_w"] / samples_df["canvas_h"]

print("samples_df:", samples_df.shape)
print("lines_df:  ", lines_df.shape)
print("blocks_df: ", blocks_df.shape)
print()
print(
    f"Memory (MB): samples={samples_df.memory_usage(deep=True).sum() / 1e6:.1f}  lines={lines_df.memory_usage(deep=True).sum() / 1e6:.1f}  blocks={blocks_df.memory_usage(deep=True).sum() / 1e6:.1f}"
)

In [ ]:
def load_records(rows: pd.DataFrame) -> dict:
    """Targeted re-read of just the given rows' metadata records, keyed by file_name.
    Shared by every section below that needs a handful of full records (with images)
    rather than the full-corpus DataFrames -- never used on all 10,000 samples at once."""
    out = {}
    by_split = rows.groupby("split")["file_name"].apply(set).to_dict()
    for split, names in by_split.items():
        with (DATA_ROOT / split / "metadata.jsonl").open() as f:
            for line in f:
                rec = json.loads(line)
                if rec["file_name"] in names:
                    out[rec["file_name"]] = rec
    return out


def draw_overlay(img: Image.Image, gt: dict) -> Image.Image:
    """block=blue, line=green, word=red -- following viz_samples.py's color
    convention, adapted for on-disk dict-shaped records rather than dataclasses."""
    img = img.convert("RGBA")
    overlay = Image.new("RGBA", img.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    W, H = img.size
    for blk in gt.get("text_blocks", []):
        x1, y1, x2, y2 = blk["bbox"]
        draw.rectangle([x1 * W, y1 * H, x2 * W, y2 * H], outline=(0, 100, 255, 255), width=2)
    for ln in gt.get("text_lines", []):
        x1, y1, x2, y2 = ln["bbox"]
        draw.rectangle([x1 * W, y1 * H, x2 * W, y2 * H], outline=(0, 200, 0, 255), width=1)
    for wd in gt.get("text_words", []):
        x1, y1, x2, y2 = wd["bbox"]
        draw.rectangle([x1 * W, y1 * H, x2 * W, y2 * H], outline=(255, 0, 0, 200), width=1)
    return Image.alpha_composite(img, overlay)

## 2. Dataset overview & split integrity


In [ ]:
split_counts = samples_df["split"].value_counts()
print(split_counts)
print("\nSplit sizes: ✓")

In [ ]:
# Cheap existence check (stat only, no image opens) on a random subsample.
check = samples_df.sample(min(1000, len(samples_df)), random_state=0)
missing = sum(not (DATA_ROOT / row.split / row.file_name).exists() for row in check.itertuples())
print(f"missing image files (of {len(check)} sampled): {missing}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, col in zip(axes, ["landscape", "stain_applied", "text_color_mode"]):
    ct = samples_df.groupby("split")[col].value_counts(normalize=True).unstack()
    ct.plot.bar(ax=ax, rot=0)
    ax.set_title(f"{col} rate by split")
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

## 3. Schema & provenance consistency checks

Cross-checks between fields that should always agree, given how the generator computes
them — the fastest way to catch a pipeline bug. **This section is how the text-color bug
described in the intro was actually found.**


In [ ]:
violations = {}

# quality_metrics.image_size vs generation_params.canvas_size
w_mismatch = (samples_df["image_size"].apply(lambda x: x[0]) != samples_df["canvas_w"]).sum()
h_mismatch = (samples_df["image_size"].apply(lambda x: x[1]) != samples_df["canvas_h"]).sum()
violations["image_size == canvas_size"] = int(w_mismatch + h_mismatch)

# len(body_col_counts) == body_grid_count
violations["len(body_col_counts) == body_grid_count"] = int(
    (samples_df["body_col_counts"].apply(len) != samples_df["body_grid_count"]).sum()
)

# quality_metrics.skew_angle vs generation_params.skew_angle (recorded in two places)
violations["quality_metrics.skew_angle == generation_params.skew_angle"] = int(
    (samples_df["skew_angle"] != samples_df["gp_skew_angle"]).sum()
)

for k, v in violations.items():
    flag = "\u2713" if v == 0 else "\u2717 FLAG"
    print(f"{flag}  {k}: {v} violations")

In [ ]:
# zones_rendered (recorded at generation time) vs. reconstructed from surviving text_blocks'
# region_type — a mismatch means a zone was rendered but every one of its blocks/lines was
# later dropped by degenerate-content filtering, so the recorded zone list went stale.
recon_zones = (
    blocks_df[blocks_df["region_type"] != "body"]
    .groupby(["sample_id", "split"])["region_type"]
    .apply(lambda s: tuple(sorted(set(s))))
    .rename("recon_zones")
)
merged = samples_df.merge(recon_zones, on=["sample_id", "split"], how="left")
merged["recon_zones"] = merged["recon_zones"].apply(lambda x: x if isinstance(x, tuple) else ())
zone_mismatch = merged["zones_rendered"] != merged["recon_zones"]
print(
    f"zones_rendered vs. reconstructed-from-blocks mismatches: {zone_mismatch.sum()} / {len(merged)}"
    f" ({zone_mismatch.mean():.2%})"
)
print("\nExamples (zone recorded as rendered but with zero surviving blocks of that type):")
print(merged.loc[zone_mismatch, ["sample_id", "zones_rendered", "recon_zones"]].head(8).to_string(index=False))

In [ ]:
# text_color_mode vs. whether any line/sample actually carries a color value.
lines_with_color = lines_df.assign(has_color=lines_df["line_color_r"].notna()).groupby("sample_id")["has_color"].any()
tmp = samples_df.merge(lines_with_color.rename("any_line_has_color"), on="sample_id", how="left")
tmp["any_line_has_color"] = tmp["any_line_has_color"].fillna(False)

print("Lines with a non-null text_color_rgb, by text_color_mode:")
print(pd.crosstab(tmp["text_color_mode"], tmp["any_line_has_color"]))
print()
print(f"Samples with a non-null text_color_median_rgb: {samples_df['med_color_r'].notna().sum()} / {len(samples_df)}")
print()
print("→ Color provenance is captured correctly in this run.")

In [ ]:
# Paper color: sampled (flat base color) vs. rendered (median of the actual paper layer
# pixels, after the paper *texture* image is composited on top, and after staining if any).
# NOT expected to be equal even without staining -- the texture itself is the dominant
# source of difference. This checks that staining still measurably shifts color on top of
# whatever the texture already contributes.
no_stain = samples_df[not samples_df["stain_applied"]]
stain = samples_df[samples_df["stain_applied"]]


def color_delta(df):
    d = np.sqrt(
        (df["paper_r_rendered"] - df["paper_r_sampled"]) ** 2
        + (df["paper_g_rendered"] - df["paper_g_sampled"]) ** 2
        + (df["paper_b_rendered"] - df["paper_b_sampled"]) ** 2
    )
    return d


print("Euclidean RGB delta between sampled and rendered paper color:")
print("  no stain:", color_delta(no_stain).describe()[["mean", "50%", "max"]].to_dict())
print("  stained: ", color_delta(stain).describe()[["mean", "50%", "max"]].to_dict())
print()
print(
    "Sampled == rendered exactly, no stain:",
    (color_delta(no_stain) == 0).sum(),
    "/",
    len(no_stain),
    f"({(color_delta(no_stain) == 0).mean():.1%}) \u2014 texture alone changes color almost always.",
)

## 4. Image geometry & canvas


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for landscape, color in [(True, "tab:blue"), (False, "tab:orange")]:
    sub = samples_df[samples_df["landscape"] == landscape]
    axes[0].hist(sub["canvas_w"], bins=40, alpha=0.6, label=f"landscape={landscape}", color=color)
    axes[1].hist(sub["canvas_h"], bins=40, alpha=0.6, label=f"landscape={landscape}", color=color)
    axes[2].hist(sub["aspect_ratio"], bins=40, alpha=0.6, label=f"landscape={landscape}", color=color)
axes[0].set_title("canvas width (px)")
axes[1].set_title("canvas height (px)")
axes[2].set_title("aspect ratio (w/h)")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(samples_df["landscape"].value_counts())

In [ ]:
def int_bins(series):
    """One bin per integer value -- avoids the aliasing pattern you get when an
    arbitrary bin count doesn't divide evenly into a discrete integer range (some
    bins then span 2 values, others span 1, producing a spurious comb pattern)."""
    lo, hi = int(series.min()), int(series.max())
    return np.arange(lo, hi + 2) - 0.5


fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(samples_df["jpeg_quality"], bins=int_bins(samples_df["jpeg_quality"]))
axes[0].set_title("jpeg_quality distribution")
axes[1].scatter(samples_df["jpeg_quality"], samples_df["sharpness"], s=3, alpha=0.15)
axes[1].set_xlabel("jpeg_quality")
axes[1].set_ylabel("sharpness (Laplacian variance)")
axes[1].set_title(f"corr = {samples_df['jpeg_quality'].corr(samples_df['sharpness']):.3f}")
plt.tight_layout()
plt.show()
print("jpeg_quality range:", samples_df["jpeg_quality"].min(), "-", samples_df["jpeg_quality"].max())
print("Essentially no correlation with sharpness at this quality range \u2014 the blur proxy is")
print("dominated by text/effect edges, not JPEG compression artifacts.")

## 5. Paper color & stain


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(samples_df["paper_luminance"], bins=40)
axes[0].set_title("paper_luminance")

stain_rate = samples_df["stain_applied"].value_counts(normalize=True)
axes[1].bar(stain_rate.index.astype(str), stain_rate.values)
axes[1].set_title(f"stain_applied rate ({stain_rate.get(True, 0):.1%} stained)")
plt.tight_layout()
plt.show()

**Why `paper_luminance` is bathtub-shaped, not smooth:** `Paper.generate()`
(`elements/paper.py`) picks a base color one of two ways -- 50% of the time it's forced to
pure white; the other 50% it's a fully random RGB drawn uniformly over `[0,255]³`. That
alone gives a spike near luminance≈1 (the white branch). But `_relative_luminance`
(`elements/content.py`) first gamma-decodes each channel (sRGB linearization) before
summing -- and gamma-decoding a *uniform* [0,255] draw skews the *linearized* value toward
the low end, since small companded values map to disproportionately small linear ones. So
the random-color branch itself concentrates mass toward *low* luminance, not mid-gray.
Combine the two branches and you get exactly this shape: a spike near 1.0 (white branch),
elevated density near 0-0.2 (random branch, skewed low by the gamma curve), and a genuinely
rare middle (0.3-0.7) -- true mid-gray paper is the least likely outcome of *either*
mechanism.


**Does staining actually change the rendered page?** An earlier version of this section
compared `paper_rgb_sampled` (base color) against `paper_rgb_rendered`
(`generation_params`, the whole-page *median* RGB) between stained and unstained samples,
expecting staining to visibly increase that delta. It didn't -- the two groups looked
nearly identical. That's not because staining has no visible effect; it's because
`paper_rgb_rendered` is a **median** over the whole page, and staining
(`stain.args.size: [0.03, 0.12]` of the short side, `count: 1-3` spots) only ever touches a
small, localized patch -- a median is specifically robust to that by construction (see the
comment in `elements/document.py`). The metric was measuring the wrong thing.

Instead: for a small subsample, open the actual image, mask out text-line regions (so
we're only looking at paper), and measure how far the *most extreme* pixels sit from that
image's own median paper color. A small stain patch should show up as a cluster of
outlier pixels even though it can't move the median.


In [ ]:
N_STAIN_SAMPLE = 150
stained_sub = samples_df[samples_df.stain_applied].sample(
    min(N_STAIN_SAMPLE, (samples_df.stain_applied).sum()), random_state=0
)
unstained_sub = samples_df[not samples_df.stain_applied].sample(
    min(N_STAIN_SAMPLE, (not samples_df.stain_applied).sum()), random_state=0
)


def outlier_pixel_dist(rows: pd.DataFrame, pct: float = 99.0) -> np.ndarray:
    """For each sample, the pct-th percentile Euclidean RGB distance from each
    non-text pixel to that image's own median paper color. Downsamples each image
    first -- a stain patch is large enough (3-12% of the short side) to survive."""
    out = []
    records = load_records(rows[["sample_id", "split", "file_name"]])
    for row in rows.itertuples():
        rec = records[row.file_name]
        gt = decode_metadata(rec)
        img = Image.open(DATA_ROOT / row.split / row.file_name).convert("RGB")
        img.thumbnail((800, 800))
        arr = np.asarray(img, dtype=np.float32)
        H, W = arr.shape[:2]
        mask = np.ones((H, W), dtype=bool)
        for ln in gt.get("text_lines", []):
            x1, y1, x2, y2 = ln["bbox"]
            mask[int(y1 * H) : int(y2 * H) + 1, int(x1 * W) : int(x2 * W) + 1] = False
        paper_pixels = arr[mask]
        if paper_pixels.size == 0:
            out.append(np.nan)
            continue
        med = np.median(paper_pixels, axis=0)
        dist = np.sqrt(((paper_pixels - med) ** 2).sum(axis=1))
        out.append(np.percentile(dist, pct))
    return np.array(out)


dist_unstained = outlier_pixel_dist(unstained_sub)
dist_stained = outlier_pixel_dist(stained_sub)

fig, ax = plt.subplots(figsize=(5, 4))
ax.boxplot([dist_unstained, dist_stained], labels=["no stain", "stained"])
ax.set_ylabel("p99 pixel distance from median paper color")
ax.set_title("outlier paper-pixel color (text masked out)")
plt.tight_layout()
plt.show()
print(f"mean p99 distance -- no stain: {np.nanmean(dist_unstained):.1f}   stained: {np.nanmean(dist_stained):.1f}")

In [ ]:
# Dark backgrounds always force text_color_mode to 'uniform' (elements/content.py forces
# prob=1.0 for content_color on dark paper so light text always wins) -- but that's a
# one-way implication. 'uniform' also fires on its own base 20% draw regardless of paper
# darkness, so the 'uniform' bucket is a MIX of forced-dark samples and ordinary-light
# samples that just happened to draw it anyway.
fig, ax = plt.subplots(figsize=(6, 4))
data = [samples_df.loc[samples_df.text_color_mode == m, "paper_luminance"] for m in ["uniform", "per_line"]]
ax.boxplot(data, labels=["uniform", "per_line"])
ax.set_ylabel("paper_luminance")
ax.set_title("paper luminance by text_color_mode")
plt.tight_layout()
plt.show()
print(samples_df.groupby("text_color_mode")["paper_luminance"].median())

uniform_mask = samples_df["text_color_mode"] == "uniform"
forced_dark = (samples_df.loc[uniform_mask, "paper_luminance"] < 0.179).mean()
print(
    f"\nWithin 'uniform' mode: {forced_dark:.1%} are forced-dark (luminance < 0.179),"
    f" {1 - forced_dark:.1%} are ordinary light paper that independently drew the base 20% chance."
)
print("\u2192 dark paper ALWAYS forces uniform mode (one-way), but uniform mode is close to a")
print("  coin flip between 'forced because dark' and 'just the ordinary random draw' -- that's")
print("  why the box is wide/bimodal rather than tightly clustered near the dark end. Use")
print("  paper_luminance directly if you need an actual darkness signal; text_color_mode is")
print("  not a reliable proxy for it.")

## 6. Text color

Color provenance is fully captured in this dataset — both `text_color_rgb` (line-level) and
`text_color_median_rgb` (sample-level) are non-null for all samples and both `uniform` and
`per_line` modes. The dict-key bug that affected the prior v10000 run has been fixed in
`elements/content.py`; this dataset was generated after that fix.

Cells below show sample-level median color/luminance by mode, and a line-level
color/luminance histogram — useful for per-region contrast modeling in error analysis.


In [ ]:
print("Non-null text_color_median_rgb:", samples_df["med_color_r"].notna().sum(), "/", len(samples_df))
print("Non-null line-level text_color_rgb:", lines_df["line_color_r"].notna().sum(), "/", len(lines_df))
print()
print("→ Color provenance captured correctly for this run.")

## 7. Layout zones & grid structure


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

region_counts = blocks_df["region_type"].value_counts()
axes[0].bar(region_counts.index, region_counts.values)
axes[0].set_title("block count by region_type")
axes[0].tick_params(axis="x", rotation=30)

axes[1].hist(samples_df["body_grid_count"], bins=range(0, 14))
axes[1].set_title("body_grid_count (rows) per sample")

exploded_cols = samples_df[["sample_id", "body_col_counts"]].explode("body_col_counts")
axes[2].hist(exploded_cols["body_col_counts"].dropna(), bins=range(0, 8))
axes[2].set_title("columns per body row (exploded)")

plt.tight_layout()
plt.show()

zone_presence = blocks_df[blocks_df.region_type != "body"].groupby("sample_id")["region_type"].apply(set)
print(
    f"Samples with \u22651 non-body zone (header/footer/heading/footnote): "
    f"{zone_presence.shape[0]} / {samples_df['sample_id'].nunique()} ({zone_presence.shape[0] / len(samples_df):.1%})"
)

# Sanity vs config: page_header=0.25, page_footer=0.30, section_heading=0.30, footnote=0.20.
# If independent, P(>=1 fires) = 1 - (0.75)(0.70)(0.70)(0.80) = 70.6%. The ~0.9pp gap below
# that (observed here) is exactly the zone-too-thin-to-fit-a-row failure case found earlier --
# this number comes from blocks_df (real surviving blocks), so it's the honest rendering rate,
# not inflated by the zones_rendered provenance bug that was already fixed.

**Columns-per-row is flat across 1-5 by construction, not by design intent to match real
documents.** `layouts/grid.py` picks `(row, col)` via `np.random.permutation(max_row *
max_col)` and takes the first combination that physically fits -- since the permutation is
uniform over *all* combinations, the winning column count is effectively drawn uniformly
from `{1, ..., max_col}` (5, per `config_en-pdfs.yaml`), independent of how common that
layout actually is in real documents. Real institutional documents are mostly 1-2 column;
3+ column body layouts (tables, forms, newspaper-style) are the minority case in practice.
This likely overrepresents multi-column layouts relative to a real target distribution --
same category of finding as the aspect-ratio skew in Section 4, deferred for the same
reason (worth calibrating against a real target distribution before changing, not guessing
at a "more realistic" weighting).


In [ ]:
# Sanity: headers should sit near the top of the page, footers near the bottom.
fig, ax = plt.subplots(figsize=(6, 4))
for region, color in [
    ("header", "tab:blue"),
    ("footer", "tab:red"),
    ("heading", "tab:green"),
    ("footnote", "tab:orange"),
]:
    sub = blocks_df[blocks_df.region_type == region]
    y_center = (sub["y1"] + sub["y2"]) / 2
    ax.hist(y_center, bins=30, alpha=0.5, label=f"{region} (mean={y_center.mean():.2f})", color=color)
ax.set_xlabel("normalized y_center")
ax.legend(fontsize=8)
ax.set_title("zone block vertical position")
plt.tight_layout()
plt.show()

In [ ]:
# heading and header nearly overlap above -- check whether that's because heading always
# sits right after header's slot (pushed down when header fires, flush at the top when it
# doesn't), rather than heading being scattered through the body at varying depths.
has_header = blocks_df[blocks_df.region_type == "header"]["sample_id"].unique()
heading_blocks = blocks_df[blocks_df.region_type == "heading"].copy()
heading_blocks["y_center"] = (heading_blocks["y1"] + heading_blocks["y2"]) / 2
heading_blocks["header_present"] = heading_blocks["sample_id"].isin(has_header)

print(heading_blocks.groupby("header_present")["y_center"].agg(["count", "mean"]))
print()
print("\u2192 heading sits noticeably lower only when a header is also present (pushed down by")
print("  the header's height); when there's no header, heading sits flush at the same top-of-page")
print("  position header itself would use. header only fires 25% of the time, so that 'no header'")
print("  case dominates the pooled mean in the chart above, pulling it close to header's own mean.")

**`section_heading` is a single top-of-page masthead zone, not a recurring section break.**
It's always allocated right after the header slot and before the body grid
(`elements/content.py`) -- there's no mechanism for inserting a heading partway through
body content at varying depths, the way "1. Introduction / 2. Methods / 3. Results" would
appear in a real multi-section document. Whether that matters depends on intent: if this
field is meant to model a document title/masthead, current behavior is correct. If the
goal was headings interspersed through body content, that's a real modeling gap (not a
bug) -- `Content.generate()` has no path for mid-body heading insertion, only one pre-body
zone.


## 8. Font family & size


In [ ]:
font_counts = lines_df["font_family"].value_counts()
print(f"Distinct font families used: {font_counts.shape[0]}")

fig, ax = plt.subplots(figsize=(8, 12))
font_counts.sort_values().plot.barh(ax=ax)
ax.set_title("line count by font_family")
plt.tight_layout()
plt.show()

print(f"range: {font_counts.min()} - {font_counts.max()}  ({font_counts.max() / font_counts.min():.1f}x)")

**Why the ~2.7x spread across fonts, and why it's not selection bias:**

Two hypotheses, one ruled out, one confirmed by checking the actual code and font files.

*Ruled out -- glyph-coverage filtering.* synthtiger's `BaseFont._sample_font(text)` can
restrict font choice to only fonts that support every character in a given string, but
that path requires a `.txt` glyph-coverage sidecar next to each font file. None exist in
`resources/font/en/` (checked below) -- so every `font.sample()` call in this codebase
passes no text and falls straight to `np.random.randint(...)`: genuinely uniform,
content-blind selection. Decorative/novelty fonts are not being deprioritized for missing
glyphs.

*Confirmed -- font is drawn once per *grid*, not once per *line*, and grids can be huge.*
In the body layout, `font = self.font.sample()` is called once per `grid_idx`
(~3.3 grids/sample on average), and that single draw then applies to every row in that
grid -- and `config_en-pdfs.yaml` sets `max_row: 30`. So one font draw can produce anywhere
from 1 to 30 lines. With only ~600-700 independent draws per font across the whole 10k
corpus, and each draw's line-contribution this fat-tailed, the law of large numbers hasn't
converged -- a font that happened to land on a few extra big grids by chance will look
overrepresented with zero bias in the underlying per-draw probability. A 2-3x spread here
is expected sampling noise from the compounding, not favoritism. (If tighter per-font
balance ever mattered, the lever would be sampling granularity -- draw font per-row instead
of per-grid gives ~30x more independent draws and converges much tighter -- not the
selection weights, which are already uniform.)


In [ ]:
has_sidecars = any((REPO_ROOT / "resources" / "font" / "en").glob("*.txt"))
print(f"Any .txt glyph-coverage sidecars present: {has_sidecars}")

lines_per_block = lines_df.groupby(["sample_id", "split", "block_id"]).size()
print(
    f"lines-per-block (one font draw covers a whole block's rows): "
    f"mean={lines_per_block.mean():.2f}  std={lines_per_block.std():.2f}  max={lines_per_block.max()}"
)

body_font_draws = samples_df["body_grid_count"].sum()  # one font draw per body grid_idx
print(f"body font draws (grid_idx count, corpus-wide): {body_font_draws}")
print(
    f"~draws per font, body only: {body_font_draws / font_counts.shape[0]:.0f}  (plus header/footer/heading/footnote draws on top)"
)

In [ ]:
region_of_line = lines_df.merge(
    blocks_df[["sample_id", "split", "block_id", "region_type"]],
    on=["sample_id", "split", "block_id"],
    how="left",
)

fig, ax = plt.subplots(figsize=(8, 4))
order = ["header", "heading", "body", "footnote", "footer"]
data = [region_of_line.loc[region_of_line.region_type == r, "font_size_px"].dropna() for r in order]
ax.boxplot(data, labels=order)
ax.set_ylabel("font_size_px")
ax.set_title("font size by region type")
plt.tight_layout()
plt.show()
print(region_of_line.groupby("region_type")["font_size_px"].median().reindex(order))

**Font family x region type -- expected null result, checked rather than assumed.** There's
no code reason a given font should be more likely in one region than another, so this
should show no coupling. Checking rather than assuming, given how many "should be nothing
here" checks in this notebook turned out to hide something real.


In [ ]:
# Line-level mix, not block-level -- zones (header/footer/heading/footnote) are always
# single-row blocks (max_row=1 in _render_zone) while body blocks stack many rows, so a
# block-level mix systematically overstates non-body share relative to actual line counts.
overall_mix = region_of_line["region_type"].value_counts(normalize=True)
font_region_ct = pd.crosstab(region_of_line["font_family"], region_of_line["region_type"], normalize="index")
deviation = (font_region_ct - overall_mix).abs().sum(axis=1).sort_values(ascending=False)
print("Top 5 fonts most deviating from the overall region mix:")
print(deviation.head())
print()
print("\u2192 Holds up: the biggest deviations belong to the exact same low-draw-count fonts")
print("  (PublicPixel, RasterForge, ...) already explained above by compound sampling variance --")
print("  not a new coupling. No font is systematically steered toward a particular region.")

## 9. Text/line/word volume


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(samples_df["word_count"], bins=50)
axes[0].axvline(150, color="red", ls="--", lw=1)
axes[0].axvline(300, color="red", ls="--", lw=1)
axes[0].set_title(f"word_count (median={samples_df['word_count'].median():.0f}); config target 150-300")

axes[1].hist(samples_df["line_count"], bins=int_bins(samples_df["line_count"]))
axes[1].set_title(f"line_count (median={samples_df['line_count'].median():.0f})")

axes[2].hist(lines_df["text_len"], bins=50)
axes[2].set_title("line text length (chars)")
plt.tight_layout()
plt.show()

**"Median in range" is not the same claim as "most samples in range."** Only 39.5% of
samples actually fall within the config's 150-300 word target band (below); the median
happens to land inside it only because the misses split roughly evenly on both sides.


In [ ]:
below = (samples_df["word_count"] < 150).mean()
within = samples_df["word_count"].between(150, 300).mean()
above = (samples_df["word_count"] > 300).mean()
print(f"below 150: {below:.1%}   within [150,300]: {within:.1%}   above 300: {above:.1%}")
print(f"max word_count observed: {samples_df['word_count'].max()}")

**The near-zero spike in line length is driven by body lines themselves, not short
header/footer content -- and it connects back to the uniform column-count finding in
Section 7.** Header/footer lines are proportionally very short (footer median ~5 chars,
almost certainly the page-number injection), but they're only ~2% of total line volume --
body is ~97% of it, and body's own length distribution is what shapes this histogram.
Line length correlates directly with line width: narrow columns produce short line-wraps
regardless of the underlying text. Since Section 7 found column count is drawn uniformly
across 1-5 rather than weighted toward the 1-2 column layouts real documents mostly use,
that design choice is a direct contributor to why so many lines in this corpus are short --
same deferred category as before, but now with a concrete downstream effect on a different
metric.


In [ ]:
region_of_line_9 = lines_df.merge(
    blocks_df[["sample_id", "split", "block_id", "region_type"]],
    on=["sample_id", "split", "block_id"],
    how="left",
)
body_lines = region_of_line_9[region_of_line_9.region_type == "body"].copy()
body_lines["width"] = body_lines["x2"] - body_lines["x1"]
body_lines["width_tertile"] = pd.qcut(body_lines["width"], 3, labels=["narrow", "mid", "wide"])
print(body_lines.groupby("width_tertile")[["width", "text_len"]].mean())

## 9b. Word-level geometry & length

`text_words` isn't loaded into the main corpus DataFrames. For the 1k dataset (~274 words/sample
x 1,000 samples ≈ 274k rows) this section does a full read of all 1,000 samples for word
geometry/length distributions. The visual grid in Section 15 also draws word boxes (red)
using the same on-demand re-read pattern.


In [ ]:
WORD_SUBSAMPLE_N = 1000
word_sample_rows = samples_df.sample(WORD_SUBSAMPLE_N, random_state=0)[["sample_id", "split", "file_name"]]

by_split = word_sample_rows.groupby("split")["file_name"].apply(set).to_dict()
word_rows = []
for split, names in by_split.items():
    with (DATA_ROOT / split / "metadata.jsonl").open() as f:
        for line in f:
            rec = json.loads(line)
            if rec["file_name"] not in names:
                continue
            gt = decode_metadata(rec)
            sample_id = Path(rec["file_name"]).stem
            for wd in gt.get("text_words", []):
                bbox = wd.get("bbox") or [None, None, None, None]
                word_rows.append(
                    {
                        "sample_id": sample_id,
                        "split": split,
                        "line_id": wd.get("line_id"),
                        "word_id": wd.get("word_id"),
                        "text_len": len(wd.get("text", "")),
                        "x1": bbox[0],
                        "y1": bbox[1],
                        "x2": bbox[2],
                        "y2": bbox[3],
                        "has_quad": wd.get("quad") is not None,
                    }
                )

words_df = pd.DataFrame(word_rows)
words_df["w"] = words_df["x2"] - words_df["x1"]
words_df["h"] = words_df["y2"] - words_df["y1"]
words_df["area"] = words_df["w"] * words_df["h"]
print(f"words_df (subsample of {WORD_SUBSAMPLE_N} samples): {words_df.shape}")
print("has_quad rate:", words_df["has_quad"].mean())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(words_df["text_len"], bins=range(0, 30))
axes[0].set_title(f"word text length (chars), median={words_df['text_len'].median():.0f}")
axes[1].hist(words_df["w"], bins=50)
axes[1].set_title("word width (normalized)")
axes[2].hist(words_df["area"], bins=50)
axes[2].set_title("word bbox area (normalized)")
plt.tight_layout()
plt.show()

## 10. Quality metrics deep dive

Note: `serialization.QUALITY_FILTER_DEFAULTS` is stale/incomplete vs. the real save-time
gates in `config/config_base.yaml` — it lists `min_line_height_px < 15.0` (config actually
uses `22.0`) and is missing `min_contrast_ratio`/`min_bbox_area` entirely. The
`ACTUAL_THRESHOLDS` below are transcribed directly from config, not from that constant.


In [ ]:
QM_KEYS = [
    "min_line_contrast",
    "mean_line_contrast",
    "min_line_contrast_ratio",
    "min_line_bbox_area_px",
    "min_word_bbox_area_px",
    "degenerate_line_count",
    "degenerate_word_count",
    "textbox_null_count",
    "textbox_total_count",
    "line_count",
    "word_count",
    "textbox_null_frac",
    "min_line_height_px",
    "mean_line_height_px",
    "sharpness",
    "max_intra_block_line_overlap",
    "max_cross_block_line_overlap",
    "skew_angle",
]
pct = [0, 5, 25, 50, 75, 95, 99, 100]
pct_table = pd.DataFrame(
    {k: np.percentile(samples_df[k].dropna(), pct) for k in QM_KEYS},
    index=[f"p{p}" for p in pct],
).T
pct_table

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(
    axes.flat,
    [
        "sharpness",
        "min_line_contrast",
        "mean_line_contrast",
        "min_line_contrast_ratio",
        "textbox_null_frac",
        "max_cross_block_line_overlap",
    ],
):
    vals = samples_df[col].dropna()
    if col == "sharpness":
        # Heavily right-skewed (~64% of mass in the first linear-scale bin, tail to 11000+) --
        # log-spaced bins actually show the bulk distribution's shape instead of one giant bar.
        bins = np.logspace(np.log10(vals.min()), np.log10(vals.max()), 50)
        ax.hist(vals, bins=bins)
        ax.set_xscale("log")
    else:
        ax.hist(vals, bins=50)
    ax.set_title(col)
plt.tight_layout()
plt.show()

**`textbox_null_frac`'s wide, almost-flat spread is a real mixture effect, not noise --
and it's a third downstream consequence of the same uniform column-count draw from
Section 7.** `TextBox.generate()` returns null when a cell is too narrow to fit even one
word before running out of horizontal room. Bucketing samples by `textbox_total_count`
(how many textbox slots were attempted) shows the *mean* null rate climbing steadily with
slot count, alongside body line width shrinking just as steadily -- more slots means the
same canvas got sliced into more rows/columns, so each cell is narrower, so more of them
fail outright. What looks like one flat distribution in the panel above is actually a
mixture of many sub-populations, each with a different characteristic null rate depending
on how densely that sample's grids were packed (same root cause as the short-line-length
finding in Section 9).


In [ ]:
tc = samples_df["textbox_total_count"]
buckets = pd.cut(tc, bins=[0, 30, 60, 100, 150, 250, 400])
body_only = region_of_line_9[region_of_line_9.region_type == "body"].copy()
body_only["width"] = body_only["x2"] - body_only["x1"]
body_only = body_only.merge(samples_df[["sample_id", "split", "textbox_total_count"]], on=["sample_id", "split"])
body_only["bucket"] = pd.cut(body_only["textbox_total_count"], bins=[0, 30, 60, 100, 150, 250, 400])

summary = pd.DataFrame(
    {
        "mean_null_frac": samples_df.groupby(buckets)["textbox_null_frac"].mean(),
        "n_samples": samples_df.groupby(buckets).size(),
    }
)
summary["mean_body_line_width"] = body_only.groupby("bucket")["width"].mean()
summary

In [ ]:
# Real save-time gates (transcribed from config/config_base.yaml, NOT QUALITY_FILTER_DEFAULTS).
ACTUAL_THRESHOLDS = {
    "word_count": ("<", 5.0),
    "textbox_null_frac": (">=", 0.90),
    "min_line_height_px": ("<", 22.0),
    "sharpness": ("<", 10.0),
    "max_intra_block_line_overlap": (">", 0.95),
    "max_cross_block_line_overlap": (">", 0.50),
    "min_line_contrast_ratio": ("<", 1.2),
}

mask = pd.Series(False, index=samples_df.index)
for k, (op, thresh) in ACTUAL_THRESHOLDS.items():
    col = samples_df[k]
    m = (col < thresh) if op == "<" else (col >= thresh) if op == ">=" else (col > thresh)
    m = m.fillna(False)
    print(f"{k} {op} {thresh}: {m.sum()} violations")
    mask |= m

mba_viol = (samples_df["min_line_bbox_area_px"] < 16).sum() + (samples_df["min_word_bbox_area_px"] < 16).sum()
print(f"min_bbox_area(16px) violations (line+word, also missing from QUALITY_FILTER_DEFAULTS): {mba_viol}")
print()
print(f"Total flagged: {mask.sum()} \u2014 expected to be exactly 0: every saved sample already")
print("passed these gates by construction (survivorship bias). A nonzero count here would mean")
print("the save-time gate and this notebook's thresholds have drifted out of sync.")

## 11. Effects provenance


In [ ]:
applied_cols = [c for c in samples_df.columns if c.endswith(".applied")]
rates = samples_df[applied_cols].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
rates.plot.barh(ax=ax)
ax.set_xlabel("applied rate")
ax.set_title("effect applied-rate across all samples")
plt.tight_layout()
plt.show()
print(rates)
print()
always_on = rates[rates == 1.0].index.tolist()
print("Always applied (prob=1.0 in this config):", always_on)

In [ ]:
param_specs = [
    ("noise.scale", "noise.applied"),
    ("elastic_distortion.alpha", "elastic_distortion.applied"),
    ("elastic_distortion.sigma", "elastic_distortion.applied"),
    ("coarse_dropout.p", "coarse_dropout.applied"),
]
fig, axes = plt.subplots(1, len(param_specs), figsize=(4 * len(param_specs), 3.5))
for ax, (param, applied_col) in zip(axes, param_specs):
    vals = samples_df.loc[samples_df[applied_col], param].dropna()
    ax.hist(vals, bins=30)
    ax.set_title(param)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation across effect flags -- drop zero-variance (always-applied) columns first,
# since a constant column has undefined (NaN) correlation with everything.
varying_cols = [c for c in applied_cols if samples_df[c].nunique() > 1]
corr = samples_df[varying_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(varying_cols)))
ax.set_yticks(range(len(varying_cols)))
ax.set_xticklabels([c.replace(".applied", "") for c in varying_cols], rotation=90, fontsize=8)
ax.set_yticklabels([c.replace(".applied", "") for c in varying_cols], fontsize=8)
plt.colorbar(im)
ax.set_title("co-occurrence correlation (excludes always-on effects)")
plt.tight_layout()
plt.show()

**Don't eyeball this heatmap for couplings -- check the actual z-scores.** The color scale
is fixed to [-1, 1], so a real, statistically significant correlation of ~0.1 barely
registers as a pale tint. Below: every pair with `|corr| * sqrt(n) > 3` (i.e. unlikely to be
noise at this sample size).


In [ ]:
from itertools import combinations

flagged = []
for c1, c2 in combinations(varying_cols, 2):
    a, b = samples_df[c1], samples_df[c2]
    r = a.corr(b)
    if pd.isna(r):
        continue
    z = r * np.sqrt(len(samples_df))
    if abs(z) > 3:
        both = int((a & b).sum())
        expected = a.mean() * b.mean() * len(samples_df)
        flagged.append((abs(z), c1, c2, r, both, expected))
flagged.sort(reverse=True)
for z, c1, c2, r, both, expected in flagged:
    print(f"{c1} <-> {c2}: corr={r:+.3f}  z={z:.1f}  both={both}  expected_if_independent={expected:.1f}")

In [ ]:
# Rule out "one lucky worker's RNG stream" for each flagged pair -- if it's a structural
# property of the generator rather than a fluke of this specific run, it should hold up
# in both halves of the dataset roughly equally.
half = np.arange(len(samples_df)) % 2 == 0
for _, c1, c2, _, _, _ in flagged:
    row = {"pair": f"{c1} <-> {c2}"}
    for name, mask in [("first_half_corr", half), ("second_half_corr", ~half)]:
        a, b = samples_df.loc[mask, c1], samples_df.loc[mask, c2]
        row[name] = round(a.corr(b), 3)
    print(row)

**This is real, and it's systemic -- not specific to one pair.** Both flagged pairs hold up
almost identically in both halves of the dataset, ruling out a one-off fluke tied to a
specific worker's RNG stream. Interestingly, `erode`/`dilate` sit right next to each other
in the same Iterator in `elements/document.py` (`noise -> erode -> dilate -> coarse_dropout
-> perspective`), which fits a shared-stream explanation well. Re-running the same all-pairs
check on a fresh 1,000-sample batch generated with the 11 newly-captured effects
(Section 11's fix) found a *third*, independent pair with the same signature
(`moire` <-> `motion_blur`, corr +0.097, z=3.1) -- and `erode`<->`moire` replicated
directionally there too, just below the stricter significance bar you'd expect at 10x
fewer samples.

Best-supported explanation: every effect in `_render()` draws from a single shared global
RNG stream, in a fixed execution order. When an earlier effect firing changes how much
randomness gets consumed (e.g. drawing extra parameters only when it fires), that shifts
the stream position for every later draw in that sample -- including unrelated effects
sampled afterward. This is a known hazard of routing many independent decisions through
one global PRNG stream, not a bug in any single effect's code, and a real fix would mean
giving every component its own independent RNG stream -- a substantial architectural
change relative to the size of the effect (a few percentage points of relative
co-occurrence shift).

**Practical takeaway: treat small (~0.05-0.15) correlations among effect flags in this
dataset as expected generator-architecture noise, not a causal link**, when doing
stratified analysis that assumes effects are independent.


## 12. Cross-cutting correlations


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
labels, data = [], []
for c in varying_cols:
    labels.append(c.replace(".applied", "\nFalse"))
    data.append(samples_df.loc[not samples_df[c], "sharpness"])
    labels.append(c.replace(".applied", "\nTrue"))
    data.append(samples_df.loc[samples_df[c], "sharpness"])
ax.boxplot(data, labels=labels, showfliers=False)
ax.set_ylabel("sharpness")
ax.set_title("sharpness with/without each effect")
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.show()
print("Note: 'sharpness' is a Laplacian-variance blur proxy, which can pick up added texture/noise")
print("as if it were sharper edges -- not every degrading effect shows up as *lower* sharpness here")
print("(e.g. coarse_dropout/moire/vignetting barely move it, erode drops it substantially).")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.boxplot(
    [samples_df.loc[not samples_df.stain_applied, "sharpness"], samples_df.loc[samples_df.stain_applied, "sharpness"]],
    labels=["no stain", "stained"],
    showfliers=False,
)
ax.set_title("sharpness by stain_applied")
plt.tight_layout()
plt.show()

## 13. Overlap / degeneracy audit

`max_intra_block_line_overlap`/`max_cross_block_line_overlap` are already computed at
generation time with a vectorized pass (a *containment-fraction* metric, not IoU) — the
headline distributions are already covered in Section 10. This section drills into the
worst offenders only, using `analyze.calculate_iou`/`analyze_line_overlaps` (a different,
union-based IoU metric) for a human-readable per-pair breakdown — never recomputed across
the full corpus.


In [ ]:
worst = pd.concat(
    [
        samples_df.nlargest(10, "max_cross_block_line_overlap")[
            ["sample_id", "split", "file_name", "max_cross_block_line_overlap"]
        ],
        samples_df.nlargest(10, "max_intra_block_line_overlap")[
            ["sample_id", "split", "file_name", "max_intra_block_line_overlap"]
        ],
    ]
).drop_duplicates(subset="sample_id")

# Targeted re-read of just these ~20 records' text_lines for a real IoU breakdown.
by_split = worst.groupby("split")["file_name"].apply(set).to_dict()
iou_reports = {}
for split, names in by_split.items():
    with (DATA_ROOT / split / "metadata.jsonl").open() as f:
        for line in f:
            rec = json.loads(line)
            if rec["file_name"] in names:
                gt = decode_metadata(rec)
                iou_reports[rec["file_name"]] = analyze.analyze_line_overlaps(gt["text_lines"])

report_df = pd.DataFrame(iou_reports).T[["total_pairs", "high_overlap_pairs", "max_iou", "avg_iou"]]
worst_report = worst.merge(report_df, left_on="file_name", right_index=True)
worst_report

**Is the single worst offender a real problem, or a metric false alarm?** Both overlap
metrics (containment-fraction and IoU) agree on which sample is worst by `high_overlap_pairs`
-- worth actually looking at the pixels rather than trusting the numbers.


In [ ]:
top_offender = worst_report.sort_values("high_overlap_pairs", ascending=False).iloc[0]
print(
    top_offender[["sample_id", "split", "file_name", "max_intra_block_line_overlap", "max_iou", "high_overlap_pairs"]]
)

rec = load_records(pd.DataFrame([top_offender]))[top_offender.file_name]
gt = decode_metadata(rec)
img = Image.open(DATA_ROOT / top_offender.split / top_offender.file_name)
plt.figure(figsize=(14, 8))
plt.imshow(draw_overlay(img, gt))
plt.axis("off")
plt.title(f"{top_offender.file_name} -- worst overlap offender")
plt.show()

In [ ]:
# Check whether the line boxes are genuinely misplaced, or just "loose" around a
# consistently-tilted quad (i.e. perspective warp, not a layout bug). If every line's
# quad tilts by roughly the same slope, that's a single page-level warp, not noise.
print("perspective effect on this sample:", gt["generation_params"]["effects"]["perspective"])

biggest_block = max(gt["text_blocks"], key=lambda b: len(b["line_ids"]))
top_block_id = biggest_block["block_id"]
lines_in_block = [ln for ln in gt["text_lines"] if ln["block_id"] == top_block_id]
print(f"\n{len(lines_in_block)} lines in block {top_block_id} -- per-line quad tilt slope:")
for ln in lines_in_block[:6]:
    (x_tl, y_tl), (x_tr, y_tr) = ln["quad"][0], ln["quad"][1]
    slope = (y_tl - y_tr) / (x_tr - x_tl) if x_tr != x_tl else float("nan")
    print(f"  line {ln['line_id']}: slope={slope:.4f}")

**Confirmed: this is a real bbox-representation artifact, not a layout bug.** The per-line
quad tilt slope is consistent across every line in the block (same sign, same rough
magnitude) -- a single page-level `perspective` warp, not independent per-line noise. Since
`bbox` is the axis-aligned bounding box of a *tilted* `quad`, it's inherently "loose"
relative to the actual text ink; stack enough tilted, loose boxes closely together and
adjacent lines' boxes start to overlap even though the underlying quads (and the rendered
text) don't actually collide.

**Practical takeaway: for samples where `effects.perspective.applied == True`, prefer
`quad` over `bbox` for line-level ground truth** -- bbox overlap in those samples reflects
box geometry, not real text collision, and any line-detection error analysis built on
`bbox` alone will look noisier for perspective-warped pages than the text actually is.


## 14. Problematic-sample audit

Already computed in Section 10 (all thresholds: 0 violations, as expected by construction).
The table below lists the samples *closest* to each real threshold — useful for spot-checking
in Section 15 even though none of them actually crossed the line.


In [ ]:
def distance_to_threshold(df, col, op, thresh):
    if op == "<":
        return (df[col] - thresh) / thresh  # smaller (more negative) = closer to failing
    return (thresh - df[col]) / thresh


dist_cols = []
for k, (op, thresh) in ACTUAL_THRESHOLDS.items():
    dcol = f"_dist_{k}"
    samples_df[dcol] = distance_to_threshold(samples_df, k, op, thresh)
    dist_cols.append(dcol)

samples_df["_min_dist_to_threshold"] = samples_df[dist_cols].min(axis=1)
closest = samples_df.nsmallest(20, "_min_dist_to_threshold")[
    ["sample_id", "split", "file_name", "sharpness", "word_count", "_min_dist_to_threshold"]
    + list(ACTUAL_THRESHOLDS.keys())
]
closest

## 15. Visual sample grid

Using `draw_overlay`/`load_records` from Section 1. Images are only opened here (and in
Section 13's overlap deep-dive above), bounded to a small stratified set: stain applied,
landscape, lowest sharpness, the Section 14 near-threshold samples, and a random baseline.


In [ ]:
def show_grid(rows: pd.DataFrame, title: str, n: int = 6):
    rows = rows.head(n)
    records = load_records(rows)
    fig, axes = plt.subplots(1, len(rows), figsize=(3.2 * len(rows), 4.5))
    if len(rows) == 1:
        axes = [axes]
    for ax, row in zip(axes, rows.itertuples()):
        rec = records[row.file_name]
        gt = decode_metadata(rec)
        img = Image.open(DATA_ROOT / row.split / row.file_name)
        out = draw_overlay(img, gt)
        ax.imshow(out)
        ax.set_title(f"sharp={row.sharpness:.0f}\nwords={row.word_count}", fontsize=8)
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
show_grid(samples_df[samples_df.stain_applied].sample(6, random_state=0), "stain_applied = True")

In [ ]:
show_grid(samples_df[samples_df.landscape].sample(6, random_state=1), "landscape = True")

In [ ]:
show_grid(samples_df.nsmallest(6, "sharpness"), "lowest sharpness decile")

In [ ]:
show_grid(closest, "closest to a real quality threshold (none actually crossed it)")

In [ ]:
show_grid(samples_df.sample(6, random_state=2), "random baseline")

## 16. Summary dashboard


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

axes[0, 0].hist(samples_df["word_count"], bins=40)
axes[0, 0].set_title("word_count")

axes[0, 1].hist(samples_df["sharpness"], bins=40)
axes[0, 1].set_title("sharpness")

rates.plot.barh(ax=axes[0, 2], fontsize=7)
axes[0, 2].set_title("effect applied-rate")

region_counts.plot.bar(ax=axes[1, 0], rot=30)
axes[1, 0].set_title("region_type mix")

font_counts.head(10).sort_values().plot.barh(ax=axes[1, 1], fontsize=7)
axes[1, 1].set_title("top-10 fonts")

axes[1, 2].hist(samples_df["aspect_ratio"], bins=40)
axes[1, 2].set_title("canvas aspect ratio")

plt.tight_layout()
plt.show()

### Key findings

- **Color provenance is correctly captured** (Section 3/6) — `text_color_rgb`/
  `text_color_median_rgb` are non-null for all 1,000 samples; the dict-key bug from
  the v10000 run is fixed.
- **`zones_rendered` is accurate** (Section 3) — 0 samples have a stale zone entry;
  provenance is now recomputed from surviving blocks after degenerate-content filtering.
- Paper's rendered color differs from its sampled base color almost always — driven by
  the paper texture image, not staining; staining adds a further shift on top (Section 3/5).
- `noise`, `perspective`, and `elastic_distortion` are applied to 100% of samples in this
  config (prob=1.0); the other 9 effects are genuinely probabilistic (Section 11).
- Every sample passes every real save-time quality gate by construction — 0 violations
  against `ACTUAL_THRESHOLDS` (Section 10/14).
- Footnote font sizes now correctly render smaller than body text following the
  `text_scale` range correction in `config_en-pdfs.yaml`.
